# 1. Libraries

## 1.1 Own Modules

In [105]:
from modules.pdf_parser import extract_text_from_static_pdf
from modules.text_preprocessor import preprocess_text
from modules.qa_engine import generate_answer
from config.settings import TOGETHER_API_KEY, MODEL_NAME_1, MODEL_NAME_2, MODEL_NAME_3, MODEL_NAME_4

## 1.2 Python modules

In [106]:
# Streamlit
import streamlit as st

# Chunking
import fitz

# Excel
import openpyxl

# Testing
import pandas as pd
from typing import Union, IO
import requests
import time
import json

# Logging
import logging

# Variation Score
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Cleaning expected keywords
import ast

# 2. Setup

## 2.1 File Name

In [107]:
file_path = "./tests/sample_docs/Sanjay_Prabhu_Kunjibettu_Resume.pdf"

## 2.2 Cosine Score setup

In [108]:
# Load once globally
embedder = SentenceTransformer('all-MiniLM-L6-v2')

## 2.3 Models

In [109]:
models = [MODEL_NAME_1, MODEL_NAME_2, MODEL_NAME_3]

## 2.4 Test Suite Dataframe

In [110]:
df_test_suite_1 = pd.read_excel("./tests/test_suite.xlsx", sheet_name="Test1", engine="openpyxl")

In [111]:
df_test_suite_2 = pd.read_excel("./tests/test_suite.xlsx", sheet_name="Test2", engine="openpyxl")

# 3. Functions

## Cleaning Keywords

In [112]:
def clean_expected_keywords_list(raw_keywords):
    cleaned = []
    for kw in raw_keywords:
        if isinstance(kw, tuple):
            # Take first element if tuple
            kw = kw[0]
        # Convert to string just in case, and strip whitespace
        cleaned.append(str(kw).strip())
    return cleaned

## 3.1 Cosine Similarity

In [113]:
def semantic_similarity(text1: str, text2: str) -> float:
    embeddings = embedder.encode([text1, text2])
    similarity = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    return round(similarity, 4)

## 3.2 Metadata Testing: Chunking, Token Size

In [114]:
def prepare_context(file_path, num_chunks=15):
    start = time.time()
    text = extract_text_from_static_pdf(file_path)
    chunks = preprocess_text(text)
    end = time.time()

    context = "\n".join(chunks[:num_chunks])
    metadata = {
        "Chunking Time (s)": round(end - start, 2),
        "Total Chunks": len(chunks),
        "Total Characters": len(text)
    }
    return context, metadata

## 3.3 Generate/Score Response

In [115]:
def generate_and_score_response(prompt, model, expected_keywords, reference_response=None, context=None):
    start = time.time()
    response = generate_answer(context, prompt, model)
    end = time.time()

    response_time = round(end - start, 2)
    matched_keywords = [kw for kw in expected_keywords if kw.lower() in response.lower()]
    consistency_score = round(len(matched_keywords) / len(expected_keywords), 2) if expected_keywords else 0.0
    similarity_score = round(semantic_similarity(response, reference_response), 2) if reference_response else 1.0

    return {
        "Prompt": prompt,
        "Response": response,
        "Response Time (s)": response_time,
        "Consistency Score": consistency_score,
        "Similarity to Baseline": similarity_score
    }

## 3.4 Batch Processing

In [116]:
def process_batch(batch_df, model, iteration, results_df, context, delay=10):
    baseline_row = batch_df.iloc[0]
    variation_rows = batch_df.iloc[1:3]

    prompt_baseline = baseline_row["question"]
    expected_keywords = baseline_row["expected_keywords"]

    if isinstance(expected_keywords, str):
        # maybe parse if string representation of list
        try:
            expected_keywords = ast.literal_eval(expected_keywords)
        except Exception:
            expected_keywords = [expected_keywords]
    elif not isinstance(expected_keywords, list):
        expected_keywords = [expected_keywords]

    expected_keywords = clean_expected_keywords_list(expected_keywords)

    # Baseline
    result = generate_and_score_response(prompt_baseline, model, expected_keywords, context=context)
    result.update({
        "Iteration": iteration,
        "Model": model,
        "Prompt Type": "Baseline"
    })
    results_df = pd.concat([results_df, pd.DataFrame([result])], ignore_index=True)
    time.sleep(delay)

    # Variations
    reference_response = result["Response"]
    for _, row in variation_rows.iterrows():
        variation_result = generate_and_score_response(row["question"], model, expected_keywords,
                                                       reference_response, context=context)
        variation_result.update({
            "Iteration": iteration,
            "Model": model,
            "Prompt Type": "Variation"
        })
        results_df = pd.concat([results_df, pd.DataFrame([variation_result])], ignore_index=True)
        time.sleep(delay)

    return results_df

## 3.5 Running Test Suite

In [117]:
def run_test_suite(df_test_suite, models, file_path, iterations=5, delay=10):
    df_results = pd.DataFrame()
    
    for iteration in range(1, iterations + 1):
        # 1. Data Preprocessing
        full_context, metadata = prepare_context(file_path)
        logging.info(f"Chunking Metadata: {metadata}")

        df_curr_results = pd.DataFrame()

        logging.info(f"--- Iteration {iteration} ---")
        iter_start = time.time()

        # 2. Batch Processing
        for i in range(0, len(df_test_suite), 3):
            batch_df = df_test_suite.iloc[i:i + 3]
            for model in models:
                batch_results = process_batch(batch_df, model, iteration, df_results, full_context, delay)
                df_curr_results = pd.concat([df_curr_results, batch_results], ignore_index=True)

        # 3. Add metadata to current iteration
        iter_time = round(time.time() - iter_start, 2)
        logging.info(f"Iteration {iteration} completed in {iter_time} seconds.")

        df_curr_results["preprocessing_metadata"] = [metadata] * len(df_curr_results)
        df_curr_results["iteration_time_log"] = [iter_time] * len(df_curr_results)

        # 4. Append to full results
        df_results = pd.concat([df_results, df_curr_results], ignore_index=True)

    return df_results

# 4. Test

## 4.1 Test 1 - Prompt Engineering PDF

In [118]:
df_results1 = run_test_suite(df_test_suite_1, models, file_path, iterations=3, delay=10)

In [119]:
df_results1.to_csv("./tests/results/test1.csv", index=False)

In [120]:
logging.info("Test results saved to ./tests/results/test1.csv")

## 4.2 Test 2 - RAG PDF

In [121]:
df_results2 = run_test_suite(df_test_suite_2, models, file_path, iterations=3, delay=10)

AttributeError: 'tuple' object has no attribute 'lower'

In [ ]:
df_results2.to_csv("./tests/results/test2.csv", index=False)

In [ ]:
logging.info("Test results saved to ./tests/results/test2.csv")